In [26]:
! pip install pymupdf python-docx openpyxl


[notice] A new release of pip is available: 26.0.1 -> 26.1.2
[notice] To update, run: pip install --upgrade pip


# DOCUMENTS LOAD

In [27]:
from pathlib import Path

def load_documents_from_dir(directory: str):

    paths = []

    for path in Path(directory).rglob("*"):

        if path.suffix.lower() in {
            ".pdf",
            ".docx",
            ".txt"
        }:
            paths.append(path)

    return paths

In [28]:
import fitz  # pymupdf

def load_pdf_as_text(path: str) -> str:
    """
    Загружает PDF и возвращает текст,
    разделённый по страницам.
    """

    doc = fitz.open(path)

    pages_text = []

    for page in doc:
        text = page.get_text("text")

        # минимальная нормализация
        text = text.replace("\r", "").strip()

        if text:
            pages_text.append(text)

    # разделяем страницы двойным переносом
    return "\n\n".join(pages_text)

In [29]:
from docx import Document

def load_docx_as_text(path: str) -> str:
    """
    Загружает DOCX и возвращает текст
    с сохранением структуры параграфов.
    """

    doc = Document(path)

    paragraphs = []

    for p in doc.paragraphs:
        text = p.text.strip()

        if not text:
            continue

        paragraphs.append(text)

    return "\n\n".join(paragraphs)

In [30]:
import os

def load_document(path: str) -> str:
    ext = os.path.splitext(path)[1].lower()

    if ext == ".pdf":
        return load_pdf_as_text(path)

    if ext == ".docx":
        return load_docx_as_text(path)

    raise ValueError(f"Unsupported format: {ext}")

# DOCUMENTS PARSING

In [31]:
from dataclasses import dataclass
from typing import Optional

@dataclass
class Chunk:
    id: str
    doc_name: str
    section: str
    text: str
    level: str  # "chapter" | "article" | "point"
    parent_id: Optional[str] = None

In [32]:
import re

PATTERNS = {
    "chapter": re.compile(r"^(Глава\s+\d+\.?.*)$", re.MULTILINE),
    "section": re.compile(r"^(Раздел\s+[IVXLCDM\d]+\.?.*)$", re.MULTILINE),
    "article": re.compile(r"^(Статья\s+\d+(\.\d+)?\.?.*)$", re.MULTILINE),
    "point": re.compile(r"^(\d+(\.\d+)*\.)\s", re.MULTILINE),
}

In [33]:
def extract_structured_headers(text: str):

    headers = []

    for level, pattern in PATTERNS.items():

        for m in pattern.finditer(text):

            headers.append({
                "level": level,
                "title": m.group(1).strip(),
                "start": m.start()
            })

    return sorted(headers, key=lambda x: x["start"])

In [34]:
LEVEL_ORDER = ["section", "chapter", "article", "point"]

def _update_stack(stack, header, node_id, level):

    # убираем всё что глубже или равно текущему уровню
    while stack:
        last_level = stack[-1]["level"]

        if LEVEL_ORDER.index(last_level) >= LEVEL_ORDER.index(level):
            stack.pop()
        else:
            break

    stack.append({
        "id": node_id,
        "level": level
    })

In [35]:
import uuid

def parse_document(doc_name: str, text: str):

    headers = extract_structured_headers(text)

    if not headers:
        return [
            Chunk(
                id=str(uuid.uuid4()),
                doc_name=doc_name,
                section="DOCUMENT",
                text=text
            )
        ]

    chunks = []
    stack = []  # хранит текущую иерархию

    for i, h in enumerate(headers):

        start = h["start"]
        end = headers[i + 1]["start"] if i + 1 < len(headers) else len(text)

        content = text[start:end].strip()

        node_id = str(uuid.uuid4())

        # определяем parent
        parent_id = stack[-1]["id"] if stack else None

        chunk = Chunk(
            id=node_id,
            doc_name=doc_name,
            level=h["level"],
            section=h["title"],
            text=content,
            parent_id=parent_id
        )

        chunks.append(chunk)

        # обновляем стек по уровню
        _update_stack(stack, h, node_id, h["level"])

    return chunks

In [36]:
def parse_directory(directory: str):

    all_chunks = []

    paths = load_documents_from_dir(directory)

    for path in paths:

        print(f"Parsing {path}")

        try:

            text = load_document(str(path))

            chunks = parse_document(
                doc_name=path.name,
                text=text
            )

            all_chunks.extend(chunks)

        except Exception as e:

            print(
                f"Failed {path}: {e}"
            )

    return all_chunks

In [37]:
def expand_path(chunk, chunks_by_id):

    path = []
    seen_levels = set()

    current = chunk

    while current:

        # берём только 1 узел каждого уровня
        if current.level not in seen_levels:

            path.append(current)
            seen_levels.add(current.level)

        if not current.parent_id:
            break

        current = chunks_by_id.get(current.parent_id)

    return list(reversed(path))

In [38]:
def build_chunks_by_id(chunks):

    return {
        chunk.id: chunk
        for chunk in chunks
    }

In [39]:
from collections import defaultdict

def build_children_by_id(chunks):

    children = defaultdict(list)

    for chunk in chunks:

        if chunk.parent_id:

            children[
                chunk.parent_id
            ].append(chunk.id)

    return dict(children)

# SPARSE RETRIEVAL

In [42]:
import re
import pymorphy3
from rank_bm25 import BM25Okapi

morph = pymorphy3.MorphAnalyzer()

def normalize(text):

    words = re.findall(
        r"\w+",
        text.lower()
    )

    return [
        morph.parse(word)[0].normal_form
        for word in words
    ]

In [43]:
! pip install pymorphy3 rank-bm25


[notice] A new release of pip is available: 26.0.1 -> 26.1.2
[notice] To update, run: pip install --upgrade pip


In [44]:
def sparse_search(query, bm25, chunks, k):

    tokens = normalize(query)

    scores = bm25.get_scores(tokens)

    ranked = sorted(
        enumerate(scores),
        key=lambda x: x[1],
        reverse=True
    )

    results = []

    for idx, score in ranked[:k]:

        results.append({
            "score": float(score),
            "chunk": chunks[idx]
        })

    return results

# DENSE RETRIEVAL

In [45]:
! pip install sentence-transformers


[notice] A new release of pip is available: 26.0.1 -> 26.1.2
[notice] To update, run: pip install --upgrade pip


In [46]:
from sentence_transformers import SentenceTransformer
import numpy as np

model = SentenceTransformer("intfloat/multilingual-e5-small")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

In [47]:
def embed_query(text: str):
    return model.encode(
        f"query: {text}",
        normalize_embeddings=True
    )


def embed_doc(text: str):
    return model.encode(
        f"passage: {text}",
        normalize_embeddings=True
    )

In [48]:
def build_dense_index(chunks):

    texts = [
        f"passage: {c.section}\n{c.text}"
        for c in chunks
    ]

    embeddings = model.encode(
        texts,
        normalize_embeddings=True,
        batch_size=32,
        show_progress_bar=True
    )

    return np.array(embeddings)

In [49]:
def dense_search(query, chunks, dense_index, k=5):

    chunk_embeddings = dense_index

    q = embed_query(query)

    scores = np.dot(chunk_embeddings, q)

    top_idx = np.argsort(scores)[::-1][:k]

    return [
        {
            "score": float(scores[i]),
            "chunk": chunks[i]
        }
        for i in top_idx
    ]

In [50]:
chunks = parse_directory(
    "./documents"
)

chunks_by_id = build_chunks_by_id(
    chunks
)

children_by_id = build_children_by_id(
    chunks
)

dense_index = build_dense_index(
    chunks
)

tokenized_chunks = [
    normalize(
        c.section + "\n" + c.text
    )
    for c in chunks
]

bm25 = BM25Okapi(
    tokenized_chunks
)

Parsing documents/personal_data.pdf


Batches:   0%|          | 0/5 [00:00<?, ?it/s]

# TOOLS

In [51]:
def rrf(rank, k=60):
    return 1.0 / (k + rank)

def hybrid_search(
    query,
    bm25,
    chunks,
    dense_index,
    top_k
):

    sparse = sparse_search(
        query = query,
        bm25=bm25,
        chunks=chunks,
        k=top_k
    )

    dense = dense_search(
        query=query,
        chunks=chunks,
        dense_index=dense_index,
        k=top_k
    )

    scores = {}

    for rank, item in enumerate(
        sparse,
        start=1
    ):

        cid = item["chunk"].id

        scores[cid] = (
            scores.get(cid, 0)
            + rrf(rank)
        )

    for rank, item in enumerate(
        dense,
        start=1
    ):

        cid = item["chunk"].id

        scores[cid] = (
            scores.get(cid, 0)
            + rrf(rank)
        )

    ranked = sorted(
        scores.items(),
        key=lambda x: x[1],
        reverse=True
    )

    results = []

    for cid, score in ranked[:top_k]:

        results.append({
            "chunk": chunks_by_id[cid],
            "score": score
        })

    return results

In [52]:
def retrieve(query: str, top_k: int = 5):

    results = hybrid_search(
        query=query,
        bm25=bm25,
        chunks=chunks,
        dense_index=dense_index,
        top_k=top_k
    )

    return [
        {
            "chunk_id": r["chunk"].id,
            "section": r["chunk"].section,
            "score": round(r["score"], 4)
        }
        for r in results
    ]

In [53]:
def read_chunk(chunk_id: str):

    chunk = chunks_by_id.get(chunk_id)

    if chunk is None:
        return {
            "error": "chunk not found"
        }

    return {
        "chunk_id": chunk.id,
        "section": chunk.section,
        "text": chunk.text
    }

In [54]:
def expand_path(chunk_id: str):

    chunk = chunks_by_id.get(chunk_id)

    if chunk is None:
        return []

    chapter = None
    article = None
    point = None

    current = chunk

    while current:

        if current.level == "chapter":
            chapter = current.section

        elif current.level == "article":
            article = current.section

        elif current.level == "point":
            point = current.section

        if not current.parent_id:
            break

        current = chunks_by_id.get(
            current.parent_id
        )

    result = []

    if chapter:
        result.append(chapter)

    if article:
        result.append(article)

    if point:
        result.append(point)

    return result

In [55]:
def get_neighbors(
    chunk_id: str,
    window: int = 1
):

    chunk = chunks_by_id.get(chunk_id)

    if chunk is None:
        return []

    parent_id = chunk.parent_id

    if parent_id is None:
        return []

    siblings = children_by_id.get(
        parent_id,
        []
    )

    try:
        idx = siblings.index(chunk_id)

    except ValueError:
        return []

    start = max(
        0,
        idx - window
    )

    end = min(
        len(siblings),
        idx + window + 1
    )

    result = []

    for cid in siblings[start:end]:

        c = chunks_by_id[cid]

        result.append({
            "chunk_id": c.id,
            "section": c.section,
            "text": c.text
        })

    return result

In [56]:
TOOLS = {
    "retrieve": retrieve,
    "read_chunk": read_chunk,
    "expand_path": expand_path,
    "get_neighbors": get_neighbors,
}

In [57]:
import json

def execute_tool(tool_call):

    name = tool_call.function.name

    arguments = json.loads(
        tool_call.function.arguments
    )

    tool = TOOLS[name]

    return tool(**arguments)

# ReAct CYCLE

In [ ]:
SYSTEM_PROMPT = """
Ты агент по нормативным документам.

Для ответа используй инструменты.

Алгоритм:
1. Сначала вызови retrieve().
2. Затем expand_path().
3. Затем read_chunk().
4. При необходимости вызови get_neighbors().
5. После получения достаточной информации дай ответ.

ВАЖНО:
Всегда отвечай ТОЛЬКО на русском языке.
Никогда не используй китайский, английский или другие языки.
Все ответы, рассуждения и объяснения должны быть исключительно на русском языке.
Если цитируешь документ, сохраняй язык оригинала документа.

Никогда не придумывай факты.
Всегда опирайся на результаты инструментов.
"""

In [59]:
tool_schemas = [
    {
        "type": "function",
        "function": {
            "name": "retrieve",
            "description": (
                "Search relevant chunks."
            ),
            "parameters": {
                "type": "object",
                "properties": {
                    "query": {
                        "type": "string"
                    }
                },
                "required": ["query"]
            }
        }
    },

    {
        "type": "function",
        "function": {
            "name": "read_chunk",
            "description": (
                "Read full chunk text."
            ),
            "parameters": {
                "type": "object",
                "properties": {
                    "chunk_id": {
                        "type": "string"
                    }
                },
                "required": ["chunk_id"]
            }
        }
    },

    {
        "type": "function",
        "function": {
            "name": "expand_path",
            "description": (
                "Get chapter/article/point path."
            ),
            "parameters": {
                "type": "object",
                "properties": {
                    "chunk_id": {
                        "type": "string"
                    }
                },
                "required": ["chunk_id"]
            }
        }
    },

    {
        "type": "function",
        "function": {
            "name": "get_neighbors",
            "description": (
                "Get neighboring chunks."
            ),
            "parameters": {
                "type": "object",
                "properties": {
                    "chunk_id": {
                        "type": "string"
                    },
                    "window": {
                        "type": "integer"
                    }
                },
                "required": ["chunk_id"]
            }
        }
    }
]

In [114]:
def run_agent(
    client,
    t : int,
    question: str,
    max_steps: int = 10
):

    messages = [
        {
            "role": "system",
            "content": SYSTEM_PROMPT
        },
        {
            "role": "user",
            "content": question
        }
    ]

    for step in range(max_steps):

        response = client.chat.completions.create(
            model="qwen2.5:7b",
            messages=messages,
            tools=tool_schemas,
            temperature = t
        )

        msg = response.choices[0].message

        if not msg.tool_calls:

            return msg.content

        messages.append(msg)

        for tc in msg.tool_calls:

            result = execute_tool(tc)

            messages.append(
                {
                    "role": "tool",
                    "tool_call_id": tc.id,
                    "content": json.dumps(
                        result,
                        ensure_ascii=False
                    )
                }
            )

    return msg.content

In [ ]:
! pip install openai

In [100]:
from openai import OpenAI

client = OpenAI(
    base_url="http://localhost:11434/v1",
    api_key="ollama"
)

In [128]:
result = run_agent(client=client, question="моя кошка является субъектом персональных прав?", max_steps=5, t = 1)

In [129]:
k = 75
for i in range(0, len(result), k):
    start = i
    end = min(i + k, len(result))
    print(result[start : end])

Для ответа на этот вопрос, нужно больше информации о юрисдикции и типе прав
, которые вы имеете в виду. Например, права на личную информацию обычно кас
аются людей или компаний.

Но если речь идет о правовой защите животного, м
огу сказать несколько общих вещей:

1. В некоторых юрисдикциях существуют з
аконы об охране животных, которые могут защитить интересы владельцев животн
ых.
2. С точки зрения законодательства России, домашние животные не являютс
я субъектами прав в обычном понимании этого термина. Несмотря на это, у соб
ственников есть обязанности и права по отношению к своим питомцам.

Чтобы д
ать более точный ответ, можно рассмотреть специальное законодательство или 
международные соглашения в этой области. 

Для этого я буду использовать ин
струменты, чтобы найти соответствующую информацию.

